In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
loaded_data=TextLoader("sample.txt",encoding="utf-8")
docs=loaded_data.load()
text_splitters=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n","\n"," ",""]
)
chunks=text_splitters.split_documents(docs)

In [4]:
embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [5]:
llm=ChatOpenAI(
    model="gpt-3.5-turbo"
)

In [6]:
persist_directory="chromaDB"
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

In [7]:
from langchain_classic.chains import create_history_aware_retriever

In [8]:
retriever = vectorstore.as_retriever(k=3)

condense_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    (
        "human",
        "Given the conversation above, write a short standalone search query (one sentence) to find relevant info. "
        "If the question is clear on its own, use it as-is. Output only the query, nothing else.",
    ),
])

In [9]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [10]:
history_aware_retriever = create_history_aware_retriever(llm, retriever, condense_prompt)
